# EAS AI Kelas B: LLM Classification Finetuning

> Aryastaya Alaauddin 5027231082

## Imports

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
import gc
import pickle
import warnings
import os
import torch.nn.functional as F
import seaborn as sns
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.cuda.empty_cache()
print(f"Using device: {device}")

In [ ]:
class Config:
    MAX_LENGTH = 256
    BATCH_SIZE = 2
    LEARNING_RATE = 3e-5
    EPOCHS = 5
    FOLDS = 5
    DROPOUT = 0.1
    HIDDEN_DIM = 384
    GRAD_ACCUMULATION_STEPS = 16
    MODEL_NAME = 'distilbert-base-uncased'
    FREEZE_LAYERS = 4
    LABEL_SMOOTHING = 0.05
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01

config = Config()

## Exploratory Data Analysis

In [ ]:
df = pd.read_csv('/kaggle/input/llm-classification-finetuning/train.csv')
train = df.copy()

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

### Model Paling Sering Muncul

In [ ]:
top_models_a = train['model_a'].value_counts().head(10)
top_models_b = train['model_b'].value_counts().head(10)

plt.figure(figsize=(10, 10))

# Plot Top 10 Model A
plt.subplot(2, 1, 1)
top_models_a.plot(kind="barh", color="skyblue")
plt.title("Top 10 Model A")
plt.xlabel("Jumlah Kemunculan")
plt.gca().invert_yaxis()  # Model paling sering di atas

# Plot Top 10 Model B
plt.subplot(2, 1, 2)
top_models_b.plot(kind="barh", color="salmon")
plt.title("Top 10 Model B")
plt.xlabel("Jumlah Kemunculan")
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

### Distribusi Label Kemenangan

In [ ]:
win_counts = pd.DataFrame({
    "A menang": train["winner_model_a"].sum(),
    "B menang": train["winner_model_b"].sum(),
    "Tie": train["winner_tie"].sum()
}, index=["count"]).T

sns.barplot(x=win_counts.index, y=win_counts["count"], palette='mako')
plt.title("Distribusi Label Kemenangan")
plt.ylabel("Jumlah")
plt.show()

### Korelasi Panjang Respons dengan Kemenangan

In [ ]:
train['a_win'] = train['winner_model_a']
train['b_win'] = train['winner_model_b']
train['tie'] = train['winner_tie']
train['response_a_len'] = train['response_a'].apply(lambda x: len(str(x).split()))
train['response_b_len'] = train['response_b'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 5))
sns.boxenplot(data=train, x='a_win', y='response_a_len', palette='coolwarm')
plt.title("Distribusi Panjang Response A (Boxen Plot)")
plt.show()

plt.figure(figsize=(10, 5))
sns.boxenplot(data=train, x='b_win', y='response_b_len', palette='coolwarm')
plt.title("Distribusi Panjang Response B (Boxen Plot)")
plt.show()

## Fungsi-fungsi Pembantu

### Create Custom Dataset

In [ ]:
class ImprovedChatbotDataset(Dataset):
    def __init__(self, data_dict, tokenizer, max_length=512):
        self.prompts = data_dict['prompts']
        self.responses_a = data_dict['responses_a'] 
        self.responses_b = data_dict['responses_b']
        self.targets = data_dict.get('targets', None)
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.prompts)
    
    def __getitem__(self, idx):
        prompt = str(self.prompts[idx])
        response_a = str(self.responses_a[idx])
        response_b = str(self.responses_b[idx])
        
        text = f"{prompt} {self.tokenizer.sep_token} {response_a} {self.tokenizer.sep_token} {response_b}"
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors="pt",
            add_special_tokens=True
        )
        
        result = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
        }
        
        if self.targets is not None:
            result['targets'] = torch.tensor(self.targets[idx], dtype=torch.long)
        
        return result

### Model Building

In [ ]:
class LightweightChatbotModel(nn.Module):
    def __init__(self, model_name, dropout=0.1, freeze_layers=4): # Hapus hidden_dim dari argumen __init__ jika tidak digunakan untuk hal lain
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        
        if hasattr(self.backbone, 'transformer') and hasattr(self.backbone.transformer, 'layer'):
            for i, layer in enumerate(self.backbone.transformer.layer):
                if i < freeze_layers:
                    for param in layer.parameters():
                        param.requires_grad = False

        elif hasattr(self.backbone, 'encoder') and hasattr(self.backbone.encoder, 'layer'):
             for i, layer in enumerate(self.backbone.encoder.layer):
                if i < freeze_layers:
                    for param in layer.parameters():
                        param.requires_grad = False
        else:
            print(f"Warning: Could not find standard layer attribute for freezing on model {model_name}. Layers might not be frozen.")

        
        backbone_hidden_size = self.backbone.config.hidden_size
        
        self.dropout = nn.Dropout(dropout)
        
        self.classifier = nn.Sequential(
            nn.Linear(backbone_hidden_size, backbone_hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(backbone_hidden_size // 2, 3)
        )
        
        if hasattr(self.backbone, 'gradient_checkpointing_enable'):
            self.backbone.gradient_checkpointing_enable()
    
    def forward(self, input_ids, attention_mask):       
        outputs = self.backbone(input_ids, attention_mask, output_hidden_states=False)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        
        return logits

### Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss to handle class imbalance better"""
    def __init__(self, alpha=1, gamma=2, label_smoothing=0.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * ce_loss
        return focal_loss.mean()

### Model Training

In [ ]:
def train_model_advanced(model, train_loader, val_loader, optimizer, scheduler, device, fold):
    criterion = FocalLoss(alpha=1, gamma=2, label_smoothing=config.LABEL_SMOOTHING)
    best_val_loss = float('inf')
    scaler = torch.cuda.amp.GradScaler()
    patience = 2
    patience_counter = 0
    
    for epoch in range(config.EPOCHS):
        # Training
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        optimizer.zero_grad()
        
        for batch_idx, batch in enumerate(train_loader):
            try:
                input_ids = batch['input_ids'].to(device, non_blocking=True)
                attention_mask = batch['attention_mask'].to(device, non_blocking=True)
                targets = batch['targets'].to(device, non_blocking=True)
                
                with torch.cuda.amp.autocast():
                    logits = model(input_ids, attention_mask)
                    loss = criterion(logits, targets) / config.GRAD_ACCUMULATION_STEPS
                
                scaler.scale(loss).backward()
                
                # Calculate accuracy
                _, predicted = torch.max(logits.data, 1)
                train_total += targets.size(0)
                train_correct += (predicted == targets).sum().item()
                
                if (batch_idx + 1) % config.GRAD_ACCUMULATION_STEPS == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad()
                    
                    if (batch_idx + 1) % (config.GRAD_ACCUMULATION_STEPS * 8) == 0:
                        clear_memory()
                
                train_loss += loss.item() * config.GRAD_ACCUMULATION_STEPS
                
                if batch_idx % 25 == 0:
                    current_acc = 100. * train_correct / train_total if train_total > 0 else 0
                    print(f'Fold {fold}, Epoch {epoch}, Batch {batch_idx}, Loss: {loss.item():.4f}, Acc: {current_acc:.2f}%')
                    
            except RuntimeError as e:
                if "out of memory" in str(e):
                    print(f"OOM at batch {batch_idx}, clearing cache...")
                    clear_memory()
                    continue
                else:
                    raise e
        
        # Validation
        model.eval()
        val_loss = 0
        val_preds = []
        val_targets = []
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for batch in val_loader:
                try:
                    input_ids = batch['input_ids'].to(device, non_blocking=True)
                    attention_mask = batch['attention_mask'].to(device, non_blocking=True)
                    targets = batch['targets'].to(device, non_blocking=True)
                    
                    with torch.cuda.amp.autocast():
                        logits = model(input_ids, attention_mask)
                        loss = criterion(logits, targets)
                    
                    val_loss += loss.item()
                    
                    _, predicted = torch.max(logits.data, 1)
                    val_total += targets.size(0)
                    val_correct += (predicted == targets).sum().item()
                    
                    probs = torch.softmax(logits, dim=1)
                    val_preds.extend(probs.cpu().numpy())
                    val_targets.extend(targets.cpu().numpy())
                    
                except RuntimeError as e:
                    if "out of memory" in str(e):
                        print("OOM in validation, clearing cache...")
                        clear_memory()
                        continue
                    else:
                        raise e
        
        if val_preds:
            val_preds = np.array(val_preds)
            val_targets_onehot = np.eye(3)[np.array(val_targets).astype(int)]
            val_log_loss = log_loss(val_targets_onehot, val_preds, eps=1e-15)
            
            train_acc = 100. * train_correct / train_total if train_total > 0 else 0
            val_acc = 100. * val_correct / val_total if val_total > 0 else 0
            avg_train_loss = train_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)
            
            print(f'Fold {fold}, Epoch {epoch}:')
            print(f'  Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%')
            print(f'  Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%, Val Log Loss: {val_log_loss:.4f}')
            
            if val_log_loss < best_val_loss:
                best_val_loss = val_log_loss
                torch.save(model.state_dict(), f'best_model_fold_{fold}.pth')
                patience_counter = 0
                print(f'  New best model saved! Log Loss: {val_log_loss:.4f}')
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f'  Early stopping triggered after {patience} epochs without improvement')
                    break
        
        clear_memory()
    
    return best_val_loss

### Preprocess Dataset

In [ ]:
def preprocess_data_advanced(df, sample_fraction=0.9):
    """Enhanced preprocessing with better data handling"""
    
    if sample_fraction < 1.0:
        # Stratified sampling to maintain class balance
        df = df.groupby(['winner_model_a', 'winner_model_b', 'winner_tie'], group_keys=False).apply(
            lambda x: x.sample(frac=sample_fraction, random_state=42)
        ).reset_index(drop=True)
        print(f"Sampled {len(df)} rows ({sample_fraction*100}% of data)")
    
    # Create targets
    df['target'] = 0  # winner_model_a
    df.loc[df['winner_model_b'] == 1, 'target'] = 1
    df.loc[df['winner_tie'] == 1, 'target'] = 2
    
    # Better text cleaning
    df['prompt'] = df['prompt'].fillna('').astype(str)
    df['response_a'] = df['response_a'].fillna('').astype(str)
    df['response_b'] = df['response_b'].fillna('').astype(str)
    
    # Remove extremely short or long texts that might be noise
    df = df[
        (df['prompt'].str.len() > 10) & 
        (df['response_a'].str.len() > 10) & 
        (df['response_b'].str.len() > 10) &
        (df['prompt'].str.len() < 2000) & 
        (df['response_a'].str.len() < 2000) & 
        (df['response_b'].str.len() < 2000)
    ].reset_index(drop=True)
    
    print(f"After filtering: {len(df)} rows")
    print(f"Target distribution: {df['target'].value_counts().to_dict()}")
    
    return df

## Main Pipeline

### Preprocessing

In [ ]:
print("Preprocessing data...")
df = preprocess_data_advanced(df, sample_fraction=1.0)

### Initializing Tokenizer

In [ ]:
print("Initializing tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

### Cross-validation

In [ ]:
# Cross-validation
skf = StratifiedKFold(n_splits=config.FOLDS, shuffle=True, random_state=42)
fold_scores = []

### Main Training Loop

In [ ]:
for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['target'])):
    print(f"\n=== Fold {fold + 1} ===")
    clear_memory()
    
    # Prepare data dictionaries
    train_data = {
        'prompts': df.iloc[train_idx]['prompt'].values,
        'responses_a': df.iloc[train_idx]['response_a'].values,
        'responses_b': df.iloc[train_idx]['response_b'].values,
        'targets': df.iloc[train_idx]['target'].values
    }
    
    val_data = {
        'prompts': df.iloc[val_idx]['prompt'].values,
        'responses_a': df.iloc[val_idx]['response_a'].values,
        'responses_b': df.iloc[val_idx]['response_b'].values,
        'targets': df.iloc[val_idx]['target'].values
    }
    
    train_dataset = ImprovedChatbotDataset(train_data, tokenizer, config.MAX_LENGTH)
    val_dataset = ImprovedChatbotDataset(val_data, tokenizer, config.MAX_LENGTH)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=config.BATCH_SIZE, 
        shuffle=True, 
        num_workers=2,
        pin_memory=False,
        drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=config.BATCH_SIZE, 
        shuffle=False, 
        num_workers=2,
        pin_memory=False
    )
    
    # Initialize lightweight model
    model = LightweightChatbotModel(
        config.MODEL_NAME,  
        config.DROPOUT,
        config.FREEZE_LAYERS
    )
    model.to(device)
    
    # Optimizer with memory-efficient settings
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), 
        lr=config.LEARNING_RATE, 
        weight_decay=0.01,
        eps=1e-6
    )
    
    total_steps = len(train_loader) * config.EPOCHS // config.GRAD_ACCUMULATION_STEPS
    scheduler = optim.lr_scheduler.LinearLR(
        optimizer, 
        start_factor=1.0, 
        end_factor=0.1, 
        total_iters=total_steps
    )
    
    # Train with memory management
    try:
        fold_score = train_model_advanced(
            model, train_loader, val_loader, optimizer, scheduler, device, fold
        )
        fold_scores.append(fold_score)
    except RuntimeError as e:
        if "out of memory" in str(e):
            print(f"Fold {fold} failed due to OOM, skipping...")
            continue
        else:
            raise e
    
    # Cleanup
    del model, optimizer, scheduler, train_loader, val_loader
    del train_dataset, val_dataset, train_data, val_data
    clear_memory()

In [ ]:
if fold_scores:
    print(f"\nCompleted fold scores: {fold_scores}")
    print(f"Mean CV score: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores) * 2:.4f})")

### Final Model Training

In [ ]:
# Train final model
print("\nTraining final model...")
clear_memory()

final_data = {
    'prompts': df['prompt'].values,
    'responses_a': df['response_a'].values,
    'responses_b': df['response_b'].values,
    'targets': df['target'].values
}

final_dataset = ImprovedChatbotDataset(final_data, tokenizer, config.MAX_LENGTH)
final_loader = DataLoader(
    final_dataset, 
    batch_size=config.BATCH_SIZE, 
    shuffle=True, 
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

final_model = LightweightChatbotModel(
    config.MODEL_NAME, 
    config.DROPOUT,
    config.FREEZE_LAYERS
)
final_model.to(device)

#### Final Model Parameters

In [ ]:
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, final_model.parameters()), 
    lr=config.LEARNING_RATE, 
    weight_decay=0.01
)

criterion = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler()

#### Final Model Training Loop

In [ ]:
for epoch in range(config.EPOCHS):
    final_model.train()
    optimizer.zero_grad()
    
    for batch_idx, batch in enumerate(final_loader):
        try:
            input_ids = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            targets = batch['targets'].to(device, non_blocking=True)
            
            with torch.cuda.amp.autocast():
                logits = final_model(input_ids, attention_mask)
                loss = criterion(logits, targets) / config.GRAD_ACCUMULATION_STEPS
            
            scaler.scale(loss).backward()
            
            if (batch_idx + 1) % config.GRAD_ACCUMULATION_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(final_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                
                if (batch_idx + 1) % (config.GRAD_ACCUMULATION_STEPS * 4) == 0:
                    clear_memory()
            
            if batch_idx % 50 == 0:
                print(f'Final model - Epoch {epoch}, Batch {batch_idx}, Loss: {loss.item():.4f}')
                
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"OOM in final training, clearing cache...")
                clear_memory()
                continue
            else:
                raise e

## Saving Model Artifacts

In [ ]:
print("\nSaving improved model artifacts...")
torch.save(final_model.state_dict(), 'final_chatbot_model.pth')
tokenizer.save_pretrained('tokenizer')

with open('model_config.pkl', 'wb') as f:
    pickle.dump(config, f)

try:
    from transformers import AutoConfig
    
    print(f"Fetching original backbone configuration for '{config.MODEL_NAME}' from Hugging Face Hub...")
    original_backbone_config = AutoConfig.from_pretrained(config.MODEL_NAME)
    original_backbone_config.save_pretrained(".")
    print(f"Original backbone 'config.json' for '{config.MODEL_NAME}' saved to: /kaggle/working/config.json")
    
    if os.path.exists("config.json"):
        print("Verification: config.json found in /kaggle/working/")
    else:
        print("Warning: config.json was NOT found in /kaggle/working/ after saving.")

except Exception as e:
    print(f"Error saving original backbone configuration: {e}")
    print("Ensure internet is enabled during this training step to fetch the original config.")

print("\nTraining and artifact saving completed!")
clear_memory()